In [1]:
!apt-get update

# Install Java 17 JDK
!apt-get install openjdk-17-jdk

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Ign:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [3,030 kB]
Get:14 http://securit

In [2]:
!git clone https://github.com/quasylab/sibilla

Cloning into 'sibilla'...
remote: Enumerating objects: 24338, done.
remote: Counting objects: 100% (7192/7192), done.
remote: Compressing objects: 100% (1898/1898), done.
remote: Total 24338 (delta 2556), reused 6991 (delta 2387), pack-reused 17146 (from 1)
Receiving objects: 100% (24338/24338), 11.27 MiB | 14.86 MiB/s, done.
Resolving deltas: 100% (9409/9409), done.


In [3]:
!cd sibilla && ./gradlew build -x test && ./gradlew installDist
!cp -a sibilla/shell/src/dist/scripts/sibilla_py .
!cd sibilla_py && pip install .

.............10%.............20%.............30%.............40%.............50%.............60%..............70%.............80%.............90%.............100%

Welcome to Gradle 8.8!

Here are the highlights of this release:
 - Running Gradle on Java 22
 - Configurable Gradle daemon JVM
 - Improved IDE performance for large projects

For more details see https://docs.gradle.org/8.8/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)


> Starting Daemon> Starting Daemon > Connecting to Daemon> IDLE<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING [1s]> Evaluating settings<-------------> 0% INITIALIZING [2s]<-------------> 0% INITIALIZING [

In [4]:
import os
os.environ["SSHELL_PATH"]="/content/sibilla/shell/build/install/sshell/"
import sibilla


- - - - - - - - - - - - - - - - - - - - -

- - - - - -  SIBILLA IMPORTED - - - - - -

- - - - - - - - - - - - - - - - - - - - -



# **WorkStation Cluster**

This case study is based on a cluster of workstations.

The system comprises two sub-clusters with N workstations in each, connected in a star topology. The switches connecting each sub-cluster are joined by a central backbone. All components can break down and there is a single repair unit to service all components.

Study under which conditions the system guarantees that the following Quality of Service (QoS) levels are guaranteed:

minimum QoS: at least 3N/4 workstations are operational and connected via switches and backbone;

premium QoS: at least N workstations are operational and connected via switches and backbone.

In [116]:
%%writefile workstation_cluster.pm

param N = 10;
param mf = 100;
param mr = 60;

const S1 = 1;
const S2 = 1;
const B = 1;
const R = 1;


species WorkstationActive1;
species WorkstationFailed1;

species WorkstationActive2;
species WorkstationFailed2;

species SwitchActive1;
species SwitchFailed1;
species SwitchActive2;
species SwitchFailed2;

species BackboneActive;
species BackboneFailed;

species RepairUnitFree;
species RepairUnitBusy;


rule workstation_fails_1{
	WorkstationActive1 -[1.0/mf]-> WorkstationFailed1
}

rule workstation_fails_2{
	WorkstationActive2 -[1.0/mf]-> WorkstationFailed2
}

rule switch_fails_1 {
	SwitchActive1 -[1.0/mf]-> SwitchFailed1 | WorkstationFailed1
}

rule switch_fails_2 {
	SwitchActive2 -[1.0/mf]-> SwitchFailed2 | WorkstationFailed2
}

rule backbone_fails {
	BackboneActive -[1.0/mf]-> BackboneFailed | SwitchFailed1 | SwitchFailed2
}

rule start_repair_process {
	WorkstationFailed1 | WorkstationFailed2 | SwitchFailed1 | SwitchFailed2 | BackboneFailed | RepairUnitFree -[1.0/mr]-> RepairUnitBusy
}


rule repair_backbone {
	RepairUnitBusy | BackboneFailed -[1.0/mr]-> RepairUnitFree | BackboneActive
}

rule repair_switch_1 {
	RepairUnitBusy | SwitchFailed1 -[1.0/mr]-> RepairUnitFree | SwitchActive1
}

rule repair_switch_2 {
	RepairUnitBusy | SwitchFailed2 -[1.0/mr]-> RepairUnitFree | SwitchActive2
}

rule repair_workstation_1 {
	RepairUnitBusy | WorkstationFailed1 -[1.0/mr]-> RepairUnitFree | WorkstationActive1
}

rule repair_workstation_2 {
	RepairUnitBusy | WorkstationFailed2 -[1.0/mr]-> RepairUnitFree | WorkstationActive2
}

measure MinQoS = 3*((#WorkstationActive1 + #WorkstationActive2)/2)/4;
measure PremQoS = (#WorkstationActive1 + #WorkstationActive2)/2;
measure active_workstations = (#WorkstationActive1 + #WorkstationActive2);
measure active_switches = (#SwitchActive1 + #SwitchActive2);
measure active_backbone = #BackboneActive;
measure busy_repair_unit = #RepairUnitBusy;
measure free_repair_unit = #RepairUnitBusy;
measure failed_workstations = (#WorkstationFailed1 + #WorkstationFailed2);


predicate minimumQoS = ((#WorkstationActive1 + #WorkstationActive2) >= 3*N/4) && ((#SwitchActive1 + #SwitchActive2) == 2) && (#BackboneActive == 1);

predicate premiumQoS = ((#WorkstationActive1 + #WorkstationActive2) >= N) && ((#SwitchActive1 + #SwitchActive2) == 2) && (#BackboneActive == 1);

system init =
	WorkstationActive1<N> |
	WorkstationActive2<N> |
	SwitchActive1<S1> |
	SwitchActive2<S2> |
	BackboneActive<B> |
	RepairUnitFree<R>;

Overwriting workstation_cluster.pm


In [117]:
sibilla_runtime = sibilla.SibillaRuntime()
sibilla_runtime.load_module("population")
sibilla_runtime.load_from_file("workstation_cluster.pm")
sibilla_runtime.set_configuration("init")
sibilla_runtime.set_deadline(1440)
sibilla_runtime.set_dt(1)
sibilla_runtime.set_replica(100)

In [118]:
sibilla_runtime.add_measure("MinQoS")
sibilla_runtime.add_measure("PremQoS")
sibilla_runtime.add_measure("active_backbone")
sibilla_runtime.add_measure("active_switches")
sibilla_runtime.add_measure("active_workstations")
sibilla_runtime.add_measure("busy_repair_unit")
sibilla_runtime.add_measure("free_repair_unit")
sibilla_runtime.add_measure("failed_workstations")

In [119]:
simulation_result = sibilla_runtime.simulate("base_model")
simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}
simulation_result.results.keys()

 The simulation has been successfully completed


dict_keys(['#MinQoS', '#PremQoS', '#active_backbone', '#active_switches', '#active_workstations', '#busy_repair_unit', '#failed_workstations', '#free_repair_unit'])

In [120]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

print(sibilla_runtime.get_parameter('N'))
print(sibilla_runtime.get_parameter('mf'))
print(sibilla_runtime.get_parameter('mr'))

simulation_result.plot(show_sd = True)

10.0
100.0
60.0


In [ ]:
reach_minQoS = sibilla_runtime.evaluate_reachability("premiumQoS", delta=0.1, epsilon= 0.01)
print(reach_minQoS)
reach_premQoS = sibilla_runtime.evaluate_reachability("minimumQoS", delta=0.1, epsilon= 0.01)
print(reach_premQoS)

In [121]:
print('Simulation took:')
print(str(simulation_result.time_enlapsed) + 'second')
print('And it used:')
print(str(simulation_result.memory_used) + 'MiB')

Simulation took:
1.023414674999998second
And it used:
0.09375MiB


In [122]:
sibilla_runtime.clear()

# **Possible Solutions**

**High mean time between failures (MTBF) and QoS measures of the network**

In [123]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')
with sibilla_runtime as sr:
  sr.load_module("population")
  sr.load_from_file("workstation_cluster.pm")
  sr.set_configuration("init")
  sr.set_deadline(1440)
  sr.set_dt(1)
  sr.set_replica(100)

  # get all the aviable parameters name
  print('all aviable parameters name : ')
  print(sr.get_parameters())

  sr.add_measure("MinQoS")
  sr.add_measure("PremQoS")
  sr.add_measure("active_backbone")
  sr.add_measure("active_switches")
  sr.add_measure("active_workstations")
  sr.add_measure("busy_repair_unit")
  sr.add_measure("free_repair_unit")
  sr.add_measure("failed_workstations")

  sr.set_parameter("mf", 480, True, False)
  sr.set_configuration("init")

  print(sr.get_parameter('N'))
  print(sr.get_parameter('mf'))
  print(sr.get_parameter('mr'))

  simulation_result = sr.simulate("High_MTBF")

  simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}

  simulation_result.plot(show_sd = True)

all aviable parameters name : 
['N', 'mf', 'mr']
mf changed to 480current configuration : init
10.0
480.0
60.0
 The simulation has been successfully completed


In [124]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

with sibilla_runtime as sr:
  sr.load_module("population")
  sr.load_from_file("workstation_cluster.pm")
  sr.set_configuration("init")
  sr.set_deadline(1440)
  sr.set_dt(1)
  sr.set_replica(100)

  # get all the aviable parameters name
  print('all aviable parameters name : ')
  print(sr.get_parameters())

  sr.add_measure("MinQoS")
  sr.add_measure("PremQoS")
  sr.add_measure("active_backbone")
  sr.add_measure("active_switches")
  sr.add_measure("active_workstations")
  sr.add_measure("busy_repair_unit")
  sr.add_measure("free_repair_unit")
  sr.add_measure("failed_workstations")

  sr.set_parameter("mf", 60, True, False)
  #sr.set_parameter("mr", 40, True, False)
  sr.set_configuration("init")

  print(sr.get_parameter('N'))
  print(sr.get_parameter('mf'))
  print(sr.get_parameter('mr'))

  simulation_result = sr.simulate("optimum_parameters")
  simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}
  simulation_result.plot(show_sd = True)

all aviable parameters name : 
['N', 'mf', 'mr']
mf changed to 60current configuration : init
10.0
60.0
60.0
 The simulation has been successfully completed


In [125]:
sibilla_runtime.clear()

**Fast Repair: Reducing the time it takes to repair failed components.**

In [126]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

with sibilla_runtime as sr:
  sr.load_module("population")
  sr.load_from_file("workstation_cluster.pm")
  sr.set_configuration("init")
  sr.set_deadline(1440)
  sr.set_dt(1)
  sr.set_replica(100)

  # get all the aviable parameters name
  print('all aviable parameters name : ')
  print(sr.get_parameters())

  sr.add_measure("MinQoS")
  sr.add_measure("PremQoS")
  sr.add_measure("active_backbone")
  sr.add_measure("active_switches")
  sr.add_measure("active_workstations")
  sr.add_measure("busy_repair_unit")
  sr.add_measure("free_repair_unit")
  sr.add_measure("failed_workstations")

  sr.set_parameter("mr", 15, True, False)
  #sr.set_parameter("mf", 720, True, False)
  sr.set_configuration("init")

  print(sr.get_parameter('N'))
  print(sr.get_parameter('mf'))
  print(sr.get_parameter('mr'))

  simulation_result = sr.simulate("Fast_repair_time")
  simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}
  simulation_result.plot(show_sd = True)

all aviable parameters name : 
['N', 'mf', 'mr']
mr changed to 15current configuration : init
10.0
100.0
15.0
 The simulation has been successfully completed


In [127]:
sibilla_runtime.clear()

**Redundancy: Backup components that activate immediately when a failure occurs.**

In [128]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

with sibilla_runtime as sr:
  sr.load_module("population")
  sr.load_from_file("workstation_cluster.pm")
  sr.set_configuration("init")
  sr.set_deadline(1440)
  sr.set_dt(1)
  sr.set_replica(100)

  # get all the aviable parameters name
  print('all aviable parameters name : ')
  print(sr.get_parameters())

  sr.add_measure("MinQoS")
  sr.add_measure("PremQoS")
  sr.add_measure("active_backbone")
  sr.add_measure("active_switches")
  sr.add_measure("active_workstations")
  sr.add_measure("busy_repair_unit")
  sr.add_measure("free_repair_unit")
  sr.add_measure("failed_workstations")

  sr.set_parameter("N", 40, True, False)
  #sr.set_parameter("mf", 300, True, False)
  sr.set_parameter("mr", 30, True, False)
  sr.set_configuration("init")

  print(sr.get_parameter('N'))
  print(sr.get_parameter('mf'))
  print(sr.get_parameter('mr'))
  print(sr.get_parameter('S'))

  simulation_result = sr.simulate("redandent_component")
  simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}
  simulation_result.plot(show_sd = True)

all aviable parameters name : 
['N', 'mf', 'mr']
N changed to 40current configuration : init
mr changed to 30current configuration : init
40.0
100.0
30.0
0.0
 The simulation has been successfully completed


In [129]:
sibilla_runtime.clear()

# **Optimal Solution**


**Factors that improves QoS:**
*   Redundent component

*   High MTBF

*   Low reapir time

In [130]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

with sibilla_runtime as sr:
  sr.load_module("population")
  sr.load_from_file("workstation_cluster.pm")
  sr.set_configuration("init")
  sr.set_deadline(1440)
  sr.set_dt(1)
  sr.set_replica(100)

  sr.add_measure("MinQoS")
  sr.add_measure("PremQoS")
  sr.add_measure("active_backbone")
  sr.add_measure("active_switches")
  sr.add_measure("active_workstations")
  sr.add_measure("busy_repair_unit")
  sr.add_measure("free_repair_unit")
  sr.add_measure("failed_workstations")

  sr.set_parameter("N", 50, True, False)
  sr.set_parameter("mf", 720, True, False)
  sr.set_parameter("mr", 30, True, False)
  sr.set_configuration("init")

  print('all aviable parameters name : ')
  print(sr.get_parameters())

  print(sr.get_parameter('N'))
  print(sr.get_parameter('mf'))
  print(sr.get_parameter('mr'))

  simulation_result = sr.simulate("optimum_parameters")
  simulation_result.results = {"#"+k : v for k,v in simulation_result.results.items()}
  simulation_result.plot(show_sd = True)
  print(sr.evaluate_reachability("minimumQoS", delta=0.1, epsilon= 0.01))
  print(sr.evaluate_reachability("premiumQoS", delta=0.1, epsilon= 0.01))


N changed to 50current configuration : init
mf changed to 720current configuration : init
mr changed to 30current configuration : init
all aviable parameters name : 
['N', 'mf', 'mr']
50.0
720.0
30.0
 The simulation has been successfully completed


 The reachability evaluation has been successfully completed

Probability of reaching minimumQoS is 
1.0 

error prob (epsilon) :  0.01
error gap  (delta)   :  0.1

 The reachability evaluation has been successfully completed

Probability of reaching premiumQoS is 
1.0 

error prob (epsilon) :  0.01
error gap  (delta)   :  0.1

